### Load and Compute

In [1]:
import numpy as np
import pandas as pd
import os
from scipy.integrate import trapz
import glob

### Setup

In [2]:
DATA_DIR        = "../data/02_processed_combined_data"
CONDITIONS_FILE = "../data/sub_conditions.csv"
OUTPUT_DIR      = "../data/04_analysis/h3_hysteresis_results/all_subjects/"

if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

# Constants
FS             = 20
WINDOW_SECONDS = 15
STEP_SECONDS   = 1
WIN_SIZE       = int(WINDOW_SECONDS * FS)
STEP_SIZE      = int(STEP_SECONDS * FS)
BRANCH_POINT   = 205
TRANSITION_END = BRANCH_POINT + WINDOW_SECONDS # 220s

SEGMENTS_PC = {1: (9, 54), 2: (55, 205), 3: (205, 243), 4: (243, 278), 5: (279, 319)}

signals = ['Gaze_X', 'Gaze_Y', 'Pupil_Mean', 'Heart_Rate', 'GSR']


### Helper Functions

In [3]:
# --- FUNC 1: CAUSAL (forward-looking) SLIDING WINDOW ---
def sliding_window_group_isc(df_pc, df_pcm, win_size, step_size, n_boot=200):
    n_samples = min(df_pc.shape[0], df_pcm.shape[0])
    # Limit to relevant movie length
    max_time = SEGMENTS_PC[5][1] * FS + (10 * FS)
    if n_samples > max_time: n_samples = int(max_time)
    
    n_pc, n_pcm = df_pc.shape[1], df_pcm.shape[1]
    results = []

    # Right-Aligned (Causal) Loop
    for start_idx in range(0, n_samples - win_size, step_size):
        end_idx = start_idx + win_size
        time_sec = end_idx / FS
        
        # Data Slices
        w_pc = df_pc.iloc[start_idx:end_idx, :].values
        w_pcm = df_pcm.iloc[start_idx:end_idx, :].values
        
        # Bootstrap
        boot_rs = []
        for _ in range(n_boot):
            i_pc = np.random.choice(n_pc, n_pc, replace=True)
            i_pcm = np.random.choice(n_pcm, n_pcm, replace=True)
            # Correlation of means
            r = np.corrcoef(w_pc[:, i_pc].mean(1), w_pcm[:, i_pcm].mean(1))[0,1]
            boot_rs.append(r)
        
        # Fisher Mean
        z_vals = np.arctanh(np.clip(boot_rs, -0.999, 0.999))
        mean_r = np.tanh(np.mean(z_vals))
        results.append({'Time_Sec': time_sec, 'Mean_r': mean_r})
        
    return pd.DataFrame(results)

# --- FUNC 2: DECAY QUANTIFICATION ---
def quantify_decay(df_timecourse):
    stats = []
    for sig in df_timecourse['Signal'].unique():
        sub = df_timecourse[df_timecourse['Signal'] == sig].sort_values('Time_Sec')
        
        # 1. Baseline (Pre-Branch)
        base_mask = (sub['Time_Sec'] >= (BRANCH_POINT - 15)) & (sub['Time_Sec'] <= BRANCH_POINT)
        baseline_r = sub.loc[base_mask, 'Mean_r'].mean()
        
        # 2. Decay Zone (205-220s)
        decay = sub[(sub['Time_Sec'] > BRANCH_POINT) & (sub['Time_Sec'] <= TRANSITION_END)]
        
        if decay.empty: continue
        
        # Metric A: Half-Life
        target = baseline_r * 0.5
        below = decay[decay['Mean_r'] < target]
        t_half = below['Time_Sec'].iloc[0] - BRANCH_POINT if not below.empty else WINDOW_SECONDS
        
        # Metric B: Persistence (Normalized AUC)
        auc = trapz(decay['Mean_r'], decay['Time_Sec'])
        max_area = baseline_r * (decay['Time_Sec'].iloc[-1] - decay['Time_Sec'].iloc[0])
        norm_auc = auc / max_area if max_area > 0 else 0
        
        stats.append({
            'Signal': sig, 'Baseline_r': baseline_r,
            'Half_Life': t_half, 'Persistence_Score': norm_auc
        })
    return pd.DataFrame(stats)

def get_subjects_by_condition(conditions_file, target_condition='PC', exclude_prior_exposure=False):
    """
    Reads the conditions file and returns a list of subject IDs for the target condition.
    
    Parameters:
    - exclude_prior_exposure (bool): If True, removes subjects where 'Seen' == 1.
    """
    if not os.path.exists(conditions_file):
        raise FileNotFoundError(f"Conditions file not found: {conditions_file}")
        
    df = pd.read_csv(conditions_file)
    
    # Clean column names
    df.columns = [c.strip() for c in df.columns]
    
    # Check required columns
    required = ['RespondentName', 'SourceStimulus']
    if not set(required).issubset(df.columns):
        raise ValueError(f"Conditions file must contain {required} columns.")
    
    # 1. Filter by Target Condition first
    condition_df = df[df['SourceStimulus'].str.strip() == target_condition].copy()
    
    # 2. Filter by Prior Exposure (New Logic)
    if exclude_prior_exposure:
        if 'Seen' in condition_df.columns:
            # Get initial count for reporting
            n_before = len(condition_df)
            
            # Filter: Keep rows where Seen is NOT 1 (handles 0, NaN, or empty)
            # We convert to numeric to be safe against "1" (str) vs 1 (int)
            condition_df['Seen'] = pd.to_numeric(condition_df['Seen'], errors='coerce').fillna(0)
            condition_df = condition_df[condition_df['Seen'] != 1]
            
            n_excluded = n_before - len(condition_df)
            if n_excluded > 0:
                print(f"[{target_condition}] Excluded {n_excluded} subjects due to prior exposure.")
        else:
            print(f"Warning: 'exclude_prior_exposure' was True, but 'Seen' column was not found in {conditions_file}.")

    # 3. Extract IDs
    subject_ids = condition_df['RespondentName'].apply(parse_subject_id).tolist()
    
    # Remove duplicates and sort
    subject_ids = sorted(list(set(subject_ids)))
    
    return subject_ids

def parse_subject_id(respondent_name):
    """
    Extracts subject ID (e.g., 'sub001') from RespondentName string.
    Handles potential extra spaces or underscores.
    """
    # Split by underscore and take the first part
    parts = respondent_name.split('_')
    sub_id = parts[0].strip()
    return sub_id

def load_physio_data(data_dir, subject_ids, signal_type='GSR'):
    """
    Loads physiological data for a list of subjects.
    
    Args:
        data_dir (str): Path to processed combined data.
        subject_ids (list): List of subject IDs to load (e.g. ['sub001', 'sub002']).
        signal_type (str): 'GSR' or 'Heart_Rate'.
        
    Returns:
        pd.DataFrame: DataFrame with shape (n_timepoints, n_subjects).
                      Columns are subject IDs.
    """
    aligned_data = {}
    valid_subjects = []
    
    # Find all files first to handle case insensitivity (sub vs Sub)
    # We look for both '_combined_physio.csv' AND '_processed_et.csv' or similar depending on where ET is.
    # Actually, previous conversation said "Integrate Physiological Data" into combined csv.
    # Let's check if Gaze/Pupil columns are in the SAME combined file or different files.
    # The 'sub001_combined_physio.csv' we viewed earlier HAD 'Pupil_Mean', 'Gaze_X', 'Gaze_Y'.
    # So we just need to ensure we map those column names correctly.
    
    all_files = glob.glob(os.path.join(data_dir, '*combined_physio.csv'))
    file_map = {}
    for f in all_files:
        basename = os.path.basename(f)
        file_id = basename.split('_')[0].lower()
        file_map[file_id] = f
        
    lengths = []
    
    for sub_id in subject_ids:
        # Check if we have a file for this subject
        lookup_id = sub_id.lower()
        
        if lookup_id not in file_map:
            #print(f"  Warning: No data file found for subject {sub_id}")
            continue
            
        file_path = file_map[lookup_id]
        
        try:
            df = pd.read_csv(file_path)
            
            # Column Mapping fallback
            col_map = {
                'GSR': ['GSR', 'GSR_Interpolated'],
                'Heart_Rate': ['Heart_Rate', 'Heart Rate'],
                'Gaze_X': ['Gaze_X', 'Interpolated Gaze X', 'X'],
                'Gaze_Y': ['Gaze_Y', 'Interpolated Gaze Y', 'Y'],
                'Pupil_Mean': ['Pupil_Mean', 'Mean_Pupil', 'Pupil Diameter']
            }
            
            target_col = signal_type
            if signal_type in col_map:
                for possible in col_map[signal_type]:
                    if possible in df.columns:
                        target_col = possible
                        break
            
            if target_col not in df.columns:
                print(f"  Warning: Column '{signal_type}' (or variants) not found in {os.path.basename(file_path)}")
                continue
            
            signal_data = df[target_col].values
            
            # Basic cleaning: handle infinite or NaN
            if not np.isfinite(signal_data).all():
                # Interpolate linear
                signal_series = pd.Series(signal_data)
                # First interpolate internal holes
                signal_series = signal_series.interpolate(method='linear', limit_direction='both')
                # Then fill edges just in case
                signal_series = signal_series.ffill().bfill()
                signal_data = signal_series.values
                
            # If still NaNs (e.g. all NaNs), warning or skip
            if np.isnan(signal_data).any():
                #print(f"  Warning: {sub_id} {target_col} contains NaNs after interpolation. Dropping.")
                continue
                
            aligned_data[sub_id] = signal_data
            valid_subjects.append(sub_id)
            lengths.append(len(signal_data))
            
        except Exception as e:
            print(f"  Error loading {sub_id}: {e}")
            
    if not aligned_data:
        raise ValueError("No valid data loaded for any subject!")
        
    # Determine common length
    # We use the minimum length to ensure no NaNs at the end, 
    # assuming start times are aligned (which they should be from previous processing).
    # However, if min length is too short (bad subject), we might want to exclude that subject.
    # Let's verify lengths.
    
    # Histogram of lengths
    length_counts = pd.Series(lengths).value_counts().sort_index()
    #print(f"\nData Length Distribution for {signal_type}:")
    #print(length_counts)
    
    # Use the most common length or min length?
    # If we use min, we might cut good data because of one short file.
    # If we use mode, we might need to pad or trim.
    # Let's use the minimum length that is "reasonable" (e.g. at least 90% of median).
    median_len = np.median(lengths)
    threshold = 0.9 * median_len
    
    # Filter out subjects that are too short
    final_subjects = []
    clean_data_dict = {}
    
    valid_lengths = [l for l in lengths if l >= threshold]
    if not valid_lengths:
         raise ValueError("All loaded files are too short compared to median length!")
         
    target_len = min(valid_lengths) # Trim everyone to the shortest VALID length
    #print(f"Trimming data to common length: {target_len} samples.")
    
    skipped_short = 0
    for sub in valid_subjects:
        data = aligned_data[sub]
        if len(data) < target_len:
            # This subject is too short (below threshold), skip
            # print(f"  Dropping {sub}: length {len(data)} < {target_len}")
            skipped_short += 1
            continue
        
        # Trim to target_len
        clean_data_dict[sub] = data[:target_len]
        final_subjects.append(sub)
        
    if skipped_short > 0:
        print(f"Dropped {skipped_short} subjects due to insufficient data length.")
        
    #print(f"Successfully loaded {len(final_subjects)} subjects for {signal_type}.\n")
    
    # Create DataFrame
    df_final = pd.DataFrame(clean_data_dict)
    
    return df_final



### Main Analysis

In [4]:
pc_subs  = get_subjects_by_condition(CONDITIONS_FILE, target_condition = 'PC',  exclude_prior_exposure=False)
pcm_subs = get_subjects_by_condition(CONDITIONS_FILE, target_condition = 'PCM', exclude_prior_exposure=False)

# --- MAIN LOOP ---
print("Running Sliding Window Analysis...")
timecourse_results = []

for signal in signals:
    print(f"  Processing {signal}...")
    try:
        d_pc = load_physio_data(DATA_DIR, pc_subs, signal)
        d_pcm = load_physio_data(DATA_DIR, pcm_subs, signal)
        
        # Run Sliding Window
        df_res = sliding_window_group_isc(d_pc, d_pcm, WIN_SIZE, STEP_SIZE, n_boot=200)
        df_res['Signal'] = signal
        timecourse_results.append(df_res)
    except Exception as e:
        print(f"    Error: {e}")

# Save Timecourses
df_final = pd.concat(timecourse_results, ignore_index=True)
df_final.to_csv(os.path.join(OUTPUT_DIR, "hysteresis_timecourses.csv"), index=False)

# Calculate and Save Metrics
print("Calculating Decay Metrics...")
df_metrics = quantify_decay(df_final)
df_metrics.to_csv(os.path.join(OUTPUT_DIR, "hysteresis_metrics.csv"), index=False)

print("Done.")

Running Sliding Window Analysis...
  Processing Gaze_X...
  Processing Gaze_Y...
  Processing Pupil_Mean...
  Processing Heart_Rate...
  Processing GSR...
Calculating Decay Metrics...
Done.
